In [ ]:
import os
from dotenv import load_dotenv
from bip_utils import (
    Bip39SeedGenerator,
    Bip44,
    Bip44Coins,
    Bip44Changes,
)

# Constants for the derivation path to improve readability
PURPOSE = 44
COIN = Bip44Coins.ETHEREUM
ACCOUNT = 0
CHANGE = Bip44Changes.CHAIN_EXT
ADDRESS_COUNT = 20


def generate_ethereum_addresses(mnemonic: str) -> list[str]:
    """
    Generates a list of Ethereum addresses derived from a BIP-39 mnemonic.
    
    Args:
        mnemonic: The BIP-39 mnemonic phrase.
        
    Returns:
        A list of Ethereum addresses derived from the path m/44'/60'/0'/0/n
    """
    # 1. Generate Seed
    seed_bytes: bytes = Bip39SeedGenerator(mnemonic).Generate()

    # 2. Derive BIP-44 Account
    # The fluent interface creates a clean derivation path: m/44'/60'/0'/0/x
    bip44_account = (
        Bip44.FromSeed(seed_bytes, COIN)
        .Purpose()
        .Coin()
        .Account(ACCOUNT)
        .Change(CHANGE)
    )

    # 3. Generate Addresses
    # List comprehensions are generally faster and more pythonic than loops with append()
    return [
        bip44_account.AddressIndex(i).PublicKey().ToAddress()
        for i in range(ADDRESS_COUNT)
    ]


In [ ]:
if __name__ == "__main__":
    load_dotenv()
    
    # Native type hint for string or None
    mnemonic: str | None = os.getenv("MNEMONIC")

    if not mnemonic:
        print("Error: MNEMONIC environment variable not found.")
        exit(1)

    addresses = generate_ethereum_addresses(mnemonic)

    print("MetaMask Addresses:")
    for address in addresses:
        print(address)